- **학습 목표**: 공분산 행렬의 고유값분해(W1 방식)와 데이터 행렬의 SVD가 같은 주성분을 준다는 것을 수치로 검증한다.

In [1]:
import numpy as np

In [11]:
def compare_pca_eig_vs_svd(X_scaled: np.ndarray, n_components: int = 2) -> dict:
    """
    요구사항:
    - 방법 1: 공분산 행렬 (X^T X)/(n-1)의 고유값분해(np.linalg.eigh)로
      상위 n_components개 주성분 투영 결과를 구한다.
    - 방법 2: X_scaled에 np.linalg.svd(full_matrices=False)를 직접 적용해
      Vt의 상위 n_components개 행으로 투영 결과를 구한다.
    - 특이값(S)으로부터 고유값을 역산한다: eigenvalue = S**2 / (n_samples - 1)
    - 두 방법의 고유값이 np.allclose로 일치하는지 확인한다.
    - 반환값: {"eigenvalues_eig": ..., "eigenvalues_from_svd": ..., "match": bool}
    """
    cov_matrix = np.cov(X_scaled, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues_eig = eigenvalues[idx][:n_components]
    print(eigenvalues_eig)
    pca = X_scaled @ eigenvectors[:, idx][:, :n_components]
    U, S, Vt = np.linalg.svd(X_scaled, full_matrices=False)
    idx_svd = np.argsort(S)[::-1]
    S = S[idx_svd][:n_components]
    print(S)
    sorted_Vt = Vt[:,idx_svd][:,:n_components]
    n_samples = X_scaled.shape[0]
    eigenvalues_from_svd = S**2 / (n_samples - 1)
    match = np.allclose(eigenvalues_eig, eigenvalues_from_svd)
    return {"eigenvalues_eig": eigenvalues_eig, "eigenvalues_from_svd": eigenvalues_from_svd, "match": match}


In [6]:
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

In [12]:
data = load_digits()
X, y = data.data, data.target
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
compare_pca_eig_vs_svd(X,5)

[179.0069301  163.71774688 141.78843909 101.1003752   69.51316559]
[2193.11933683  566.99677184  542.00493276  504.1516975   425.59296526]


{'eigenvalues_eig': array([179.0069301 , 163.71774688, 141.78843909, 101.1003752 ,
         69.51316559]),
 'eigenvalues_from_svd': array([2678.04700757,  179.0007457 ,  163.56867881,  141.51945105,
         100.85154348]),
 'match': False}